# Flight-delay pipeline: experiment runner for Kaggle

Runs trial variants of the project's pipeline on Kaggle's machine instead of the
laptop. It clones the code from GitHub (branch `test`), rebuilds the curated table from
the `usdot/flight-delays` dataset attached to this notebook, runs the experiments listed
in **Config**, and leaves the results in `/kaggle/working/results/` for download.

**Settings this notebook needs:** Internet **on** (pip and git clone), dataset
`usdot/flight-delays` attached, accelerator none (CPU).

**Read the numbers against each other, not against the laptop.** `randomSplit` depends
on how the data is partitioned, which depends on the core count, so Kaggle's train/test
split differs from the laptop's. Every experiment set therefore includes its own
reference (`predeparture_v2`) and is compared within this run.

## Config

In [ ]:
# ---- Config -----------------------------------------------------------------
REPO_URL = "https://github.com/YN2TB/Supervised_learning_pipeline_spark.git"
BRANCH = "test"

SAMPLE = 0.01          # fraction of all flights; 0.01 is the usual ablation size
FOLDS = 2
MODELS = "all"
PREFIX = "kg-"         # MLflow experiment names start with this

# One entry per experiment: (name, extra command-line arguments). The reference
# comes first so every variant can be read against it within this run.
RUNS = [
    ("v2",              "--feature-set predeparture_v2 --arms nopca"),
    ("v2_cyclic",       "--feature-set predeparture_v2_cyclic --arms nopca"),
    ("v2_pca_all",      "--feature-set predeparture_v2 --arms pca --pca-scope all --parallelism 2"),
    ("v2_pca_numeric",  "--feature-set predeparture_v2 --arms pca --pca-scope numeric --parallelism 2"),
]

## Setup: Java, pinned packages, code

In [ ]:
import glob, os, shutil, subprocess, sys, time

WORK = "/kaggle/working"
REPO = f"{WORK}/repo"

def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-3000:])
    if check and r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")
    return r

print("python", sys.version.split()[0])
# Spark needs a Java runtime; install one if the image has none.
if sh("java -version", check=False).returncode:
    sh("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null")
    sh("java -version")
# The project pins these two (SETUP.md): MLflow 3 removed stage transitions.
sh(f"{sys.executable} -m pip install -q pyspark==3.5.4 mlflow==2.19.0")

shutil.rmtree(REPO, ignore_errors=True)
sh(f"git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
sh(f"git -C {REPO} log --oneline -1")

# Memory: give Spark's driver ~60% of the machine.
mem_gb = int(open("/proc/meminfo").read().split()[1]) // 1024 // 1024
driver_gb = max(4, int(mem_gb * 0.6))
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-memory {driver_gb}g pyspark-shell"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYTHONPATH"] = REPO
print(f"machine RAM ~{mem_gb} GB, Spark driver {driver_gb} GB, cores {os.cpu_count()}")

## Data: rebuild the curated table (~5–10 min)

In [ ]:
# The raw CSVs come from the attached dataset; the parquet tables go to /tmp so
# they are not saved as notebook output.
src = os.path.dirname(glob.glob("/kaggle/input/**/flights.csv", recursive=True)[0])
data = f"{REPO}/data"
for name in ("flights.csv", "airports.csv", "airlines.csv"):
    target = f"{data}/{name}"
    if not os.path.exists(target):
        os.symlink(f"{src}/{name}", target)
os.makedirs("/tmp/flight_parquet", exist_ok=True)
if not os.path.exists(f"{data}/parquet"):
    os.symlink("/tmp/flight_parquet", f"{data}/parquet")

t0 = time.time()
for step in ("raw", "curate"):
    r = sh(f"cd {REPO} && {sys.executable} -u data_prep.py --step {step}")
print(f"data ready in {time.time() - t0:.0f} s")

## Run the experiments

In [ ]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
for name, extra in RUNS:
    exp = PREFIX + name
    cmd = (f"cd {REPO} && {sys.executable} -u mllib_pipeline.py {extra} "
           f"--experiment {exp} --sample-fraction {SAMPLE} --tune-fraction 0.5 "
           f"--folds {FOLDS} --models {MODELS} --no-register --note kaggle_{name}")
    t0 = time.time()
    print(f"\n=== {exp}: {extra}")
    with open(f"{WORK}/logs/{exp}.log", "w") as log:
        rc = subprocess.run(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT).returncode
    tail = open(f"{WORK}/logs/{exp}.log", errors="replace").read().replace("\r", "\n").splitlines()
    for line in tail:
        if line.startswith(("  pca", "  nopca", "  baseline", "  rule", "===")) or "FAILED" in line:
            print(line)
    print(f"--- {exp} exit {rc} in {(time.time() - t0) / 60:.1f} min")

## Results (download `/kaggle/working/results/`)

In [ ]:
out = f"{WORK}/results"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)
sh(f"cd {REPO} && {sys.executable} scripts/compare_experiments.py {PREFIX} "
   f"--db {REPO}/mlflow.db --csv {out}/summary.csv")
print(open(f"{out}/summary.csv").read())
shutil.copy(f"{REPO}/mlflow.db", f"{out}/mlflow.db")
for d in glob.glob(f"{REPO}/experiments/{PREFIX}*"):
    shutil.copytree(d, f"{out}/{os.path.basename(d)}")
shutil.copytree(f"{WORK}/logs", f"{out}/logs")
sh(f"git -C {REPO} rev-parse HEAD > {out}/commit.txt")
# Keep the saved output small: the code checkout and the model artifacts stay behind.
shutil.rmtree(REPO, ignore_errors=True)
shutil.rmtree(f"{WORK}/logs", ignore_errors=True)
sh(f"du -sh {out}")